# 03 - Infill: baselines and state-of-the-art methods

Runs the reference methods on saved trials and compares them inside the lost region.

Run notebook 02 first (or `scripts/run_trials.py`).

In [ ]:
%matplotlib inline
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from hmi_infill import viz
from hmi_infill.pipeline import Trial
from hmi_infill.infill import InfillInput, run, ZeroFill, Laplace, Biharmonic, PreviousFrame, FSE, DMVE
from hmi_infill.metrics import evaluate

TRIALS = Path("../trials/notebook_demo")
trials = [Trial.load(d) for d in sorted(TRIALS.glob("frame_*"))]
scale = trials[0].meta.get("prep", {}).get("scale", 0.1)
print(len(trials), "frames;", [t.meta["blocks_lost"] for t in trials], "lost blocks per frame")

## Run every method on every frame with losses

`mask_lost` is the full region the lost coefficients touch. Swap in `mask_core` to compare.

In [ ]:
MASK = "mask_lost"
methods = [ZeroFill(), Laplace(), Biharmonic(), FSE(), PreviousFrame(), DMVE()]   # FSE = spatial SOTA, DMVE = temporal SOTA
rows, filled = [], {}
for i, t in enumerate(trials):
    if t.meta["blocks_lost"] == 0:
        continue
    x = InfillInput.from_trials(trials, i, n_past=3, n_future=1, mask=MASK)
    for m in methods:
        if isinstance(m, (PreviousFrame, DMVE)) and not x.past:
            continue
        out = run(m, x)
        filled[(i, m.name)] = out
        r = evaluate(out, t.truth, x.mask, valid=t.valid, scale=scale)
        rows.append((i, m.name, r["rmse"], r["mae"], r["psnr"], r["ssim_bbox"], r["std_ratio"]))

print(f"{'frame':>5} {'method':>15} {'RMSE':>9} {'MAE':>9} {'PSNR':>7} {'SSIM':>6} {'std ratio':>9}")
for row in rows:
    print(f"{row[0]:>5} {row[1]:>15} {row[2]:9.2f} {row[3]:9.2f} {row[4]:7.2f} {row[5]:6.3f} {row[6]:9.3f}")

## Visual comparison

In [ ]:
i = max({k[0] for k in filled}, key=lambda j: trials[j].meta["blocks_lost"])
t = trials[i]
ys, xs = np.nonzero(t.mask_core)
cy, cx, h = int(np.median(ys)), int(np.median(xs)), 64
zoom = (max(cy - h, 0), cy + h, max(cx - h, 0), cx + h)
for name in [m.name for m in methods if (i, m.name) in filled]:
    viz.compare(t.truth, t.damaged, filled[(i, name)], getattr(t, MASK), zoom=zoom, title=f"frame {i}: {name}", symmetric=True)